In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

Otsu Thresholding for Bounding Box

In [ ]:
#!/usr/bin/env python3
"""
volume_to_weight_train_gridsearch.py

Modified version:
 - Uses 3 rounds of Grid Search CV (instead of Bayesian optimization).
 - Each round runs 5 epochs to quickly evaluate hyperparameters.
 - Each round narrows the search space around the previous best.
 - Final training uses full epochs (default = 50).
 - After training, prints MSE, MAE, and R² on the test set.
"""

import os
import math
import csv
import random
from typing import List, Tuple, Dict, Any

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ----------------------------
# Utilities: Otsu thresholding
# ----------------------------
def otsu_threshold_from_array(gray: np.ndarray) -> int:
    hist, _ = np.histogram(gray.ravel(), bins=256, range=(0, 256))
    total = hist.sum()
    if total == 0:
        return 0
    prob = hist.astype(np.float64) / total
    omega = np.cumsum(prob)
    mu = np.cumsum(prob * np.arange(256))
    mu_total = mu[-1]
    denom = omega * (1.0 - omega) + 1e-12
    sigma_b2 = (mu_total * omega - mu) ** 2 / denom
    sigma_b2[omega == 0] = 0
    sigma_b2[omega == 1] = 0
    return int(np.argmax(sigma_b2))

# ----------------------------
# Mask / bbox / volume utils
# ----------------------------
def image_to_mask(img_path: str) -> np.ndarray:
    img = Image.open(img_path).convert("L")
    arr = np.array(img)
    t = otsu_threshold_from_array(arr)
    mask = (arr > t).astype(np.uint8)
    return mask

def bbox_from_mask(mask: np.ndarray) -> Tuple[int,int,int,int]:
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return (0,0,0,0)
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def avg_row_width(mask: np.ndarray, y: int, minx: int, maxx: int, half_window: int = 1) -> float:
    H, W = mask.shape
    ys = range(max(0, y-half_window), min(H, y+half_window+1))
    widths = [int(np.sum(mask[yy, minx:maxx+1] > 0)) for yy in ys]
    return float(np.mean(widths)) if widths else 0.0

def estimate_volume_from_mask(mask: np.ndarray, N_slices: int, bbox: Tuple[int,int,int,int], boundary_window: int = 1) -> float:
    minx, miny, maxx, maxy = bbox
    if minx == maxx and miny == maxy:
        return 0.0

    L = maxy - miny + 1
    if L <= 0:
        return 0.0
    slice_h = float(L) / float(N_slices)

    def area_at_row(yf: float) -> float:
        y = int(round(yf))
        y = max(miny, min(maxy, y))
        w = avg_row_width(mask, y, minx, maxx, boundary_window)
        return (math.pi / 4.0) * (w ** 2)

    total_vol = 0.0
    for i in range(N_slices):
        top_f = miny + i * slice_h
        bottom_f = miny + (i + 1) * slice_h
        h = bottom_f - top_f
        A_top = area_at_row(top_f)
        A_bottom = area_at_row(bottom_f)
        if i == 0:
            V = (h / 3.0) * A_bottom
        elif i == N_slices - 1:
            V = (h / 3.0) * A_top
        else:
            V = (h / 3.0) * (A_top + A_bottom + math.sqrt(max(0.0, A_top * A_bottom)))
        total_vol += V
    return total_vol

# ----------------------------
# CSV loader
# ----------------------------
def load_csv_pairs(csv_file: str, img_root: str = "") -> List[Tuple[str, float]]:
    pairs = []
    missing = []
    img_root_norm = os.path.normpath(img_root) if img_root else ""

    with open(csv_file, "r", newline='') as f:
        reader = csv.reader(f)
        header = next(reader, None)
        if header and len(header) >= 2:
            has_header = "image" in header[0].lower() or "weight" in header[1].lower()
        else:
            has_header = False

        def resolve_path(image_name: str) -> str:
            image_name = image_name.strip().strip('"').strip("'")
            pth = os.path.join(img_root_norm, image_name)
            if os.path.exists(pth):
                return os.path.normpath(pth)
            return ""

        if not has_header and header:
            # header actually contains the first data row
            try:
                img_p = resolve_path(header[0])
                wt = float(header[1])
                if img_p:
                    pairs.append((img_p, wt))
                else:
                    missing.append(header[0])
            except Exception:
                missing.append(header[0])

        for row in reader:
            if len(row) < 2:
                continue
            img_name = row[0]
            try:
                wt = float(row[1])
            except:
                try:
                    wt = float(row[1].strip())
                except:
                    missing.append(img_name)
                    continue
            img_p = resolve_path(img_name)
            if img_p:
                pairs.append((img_p, wt))
            else:
                missing.append(img_name)

    if missing:
        print(f"[warning] {len(missing)} image paths in CSV could not be resolved - skipped.")
    print(f"[info] Loaded {len(pairs)} valid image pairs from '{csv_file}'.")
    return pairs

# ----------------------------
# Features & model
# ----------------------------
class SimpleScaler:
    def fit(self, X):
        self.mean_, self.std_ = np.mean(X,0,keepdims=True), np.std(X,0,keepdims=True)
        self.std_[self.std_==0]=1
    def transform(self, X): return (X - self.mean_) / self.std_
    def fit_transform(self, X): self.fit(X); return self.transform(X)

def poly_features(vols: np.ndarray, degree: int) -> np.ndarray:
    vols = np.array(vols).reshape(-1,1)
    return np.column_stack([vols[:,0]**d for d in range(1, degree+1)])

class PolyRegModel(nn.Module):
    def __init__(self, in_features):
        super().__init__(); self.lin = nn.Linear(in_features, 1)
    def forward(self, x): return self.lin(x).squeeze(1)

# ----------------------------
# Training routine
# ----------------------------
def train_one_model(X_train, y_train, X_val, y_val, degree, lr, alpha, l1_ratio, batch_size, epochs, device):
    # X_train, X_val expected as 1D arrays (n,) or shape (n,1)
    X_train = np.array(X_train).reshape(-1)
    X_val = np.array(X_val).reshape(-1)
    y_train = np.array(y_train).reshape(-1)
    y_val = np.array(y_val).reshape(-1)

    X_train_f = poly_features(X_train, degree)
    X_val_f = poly_features(X_val, degree)
    scaler = SimpleScaler()
    X_train_s = scaler.fit_transform(X_train_f)
    X_val_s = scaler.transform(X_val_f)

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train_s,dtype=torch.float32), torch.tensor(y_train,dtype=torch.float32)),
        batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.tensor(X_val_s,dtype=torch.float32), torch.tensor(y_val,dtype=torch.float32)),
        batch_size=max(1,batch_size//2), shuffle=False
    )

    model = PolyRegModel(degree).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss, best_state = float('inf'), None
    for _ in range(epochs):
        model.train()
        for xb,yb in train_loader:
            xb,yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            preds = model(xb)
            mse = ((preds - yb)**2).mean()
            l1 = sum(p.abs().sum() for p in model.parameters())
            l2 = sum((p**2).sum() for p in model.parameters())
            loss = mse + alpha*(l1_ratio*l1 + (1-l1_ratio)*0.5*l2)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = 0.0
            for xb,yb in val_loader:
                xb,yb = xb.to(device), yb.to(device)
                preds = model(xb)
                mse = ((preds - yb)**2).mean()
                l1 = sum(p.abs().sum() for p in model.parameters())
                l2 = sum((p**2).sum() for p in model.parameters())
                val_loss += (mse + alpha*(l1_ratio*l1 + (1-l1_ratio)*0.5*l2)).item()
            val_loss /= max(1, len(val_loader))
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    return model, scaler, best_loss

# ----------------------------
# Metrics computation
# ----------------------------
def compute_metrics_and_preds(model, scaler, X, y, degree, device):
    X_f = poly_features(np.array(X).reshape(-1,1), degree)
    X_s = scaler.transform(X_f)
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_s,dtype=torch.float32).to(device)).cpu().numpy()
    mse = ((preds - y)**2).mean()
    mae = np.mean(np.abs(preds - y))
    ss_res = ((y - preds) ** 2).sum()
    ss_tot = ((y - np.mean(y)) ** 2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return {"mse": mse, "mae": mae, "r2": r2}, preds

# ----------------------------
# Grid search
# ----------------------------
def grid_search_cv(vols_dict: Dict[int, np.ndarray],
                   train_idx: np.ndarray, val_idx: np.ndarray, weights: np.ndarray,
                   device, batch_size, rounds=3, epochs_per_round=5) -> Dict[str, Any]:
    """
    vols_dict: mapping N_slices -> (n,1) array of volumes for all samples (order matches `weights` and `pairs`)
    train_idx, val_idx: indices into the full dataset indicating train and val splits
    weights: full weights array (length n)
    """
    # Initial broad grid
    param_grid = {
        "lr": [1e-3, 1e-2, 1e-1],
        "alpha": [1e-6, 1e-4, 1e-2],
        "l1_ratio": [0.0, 0.5, 1.0],
        "degree": [1, 2, 3, 4, 5],
        "N_slices": sorted(list(vols_dict.keys()))
    }

    best_params = None
    for r in range(rounds):
        print(f"\n--- Grid Search Round {r+1}/{rounds} ---")
        best_loss = float("inf")
        # iterate grid
        for lr in param_grid["lr"]:
            for alpha in param_grid["alpha"]:
                for l1_ratio in param_grid["l1_ratio"]:
                    for degree in param_grid["degree"]:
                        for N_slices in param_grid["N_slices"]:
                            vols_all = vols_dict.get(N_slices)
                            if vols_all is None:
                                continue
                            X_train = vols_all[train_idx].reshape(-1)
                            y_train = weights[train_idx]
                            X_val = vols_all[val_idx].reshape(-1)
                            y_val = weights[val_idx]

                            try:
                                model, scaler, vloss = train_one_model(
                                    X_train, y_train, X_val, y_val,
                                    degree, lr, alpha, l1_ratio,
                                    batch_size, epochs_per_round, device
                                )
                            except AssertionError as e:
                                # size mismatch inside train_one_model -> skip this config
                                print(f"[warn] skipped config lr={lr},alpha={alpha},l1_ratio={l1_ratio},degree={degree},N_slices={N_slices} due to error: {e}")
                                continue

                            if vloss < best_loss:
                                best_loss = vloss
                                best_params = {
                                    "lr": lr,
                                    "alpha": alpha,
                                    "l1_ratio": l1_ratio,
                                    "degree": degree,
                                    "N_slices": N_slices
                                }
        print("Best so far:", best_params, "loss:", best_loss)

        if best_params is None:
            # nothing found, return defaults
            print("[error] No valid params found in grid search; returning defaults.")
            return {
                "lr": 1e-3, "alpha": 1e-6, "l1_ratio": 0.0, "degree": 2,
                "N_slices": sorted(list(vols_dict.keys()))[0]
            }

        # Narrow grid for next round
        def narrow(values, best, is_float=False, factor=2.0):
            # For floats: create [best/factor, best, best*factor]
            if is_float:
                candidates = sorted(set([max(1e-12, best/factor), best, best*factor]))
                return candidates
            else:
                # integer-like
                best_i = int(best)
                return sorted(set([max(1,best_i-1), best_i, best_i+1]))

        param_grid = {
            "lr": narrow(param_grid["lr"], best_params["lr"], is_float=True, factor=2.0),
            "alpha": narrow(param_grid["alpha"], best_params["alpha"], is_float=True, factor=10.0),
            "l1_ratio": sorted(set([
                max(0.0, best_params["l1_ratio"]-0.25),
                best_params["l1_ratio"],
                min(1.0, best_params["l1_ratio"]+0.25)
            ])),
            "degree": narrow(param_grid["degree"], best_params["degree"], is_float=False),
            "N_slices": narrow(param_grid["N_slices"], best_params["N_slices"], is_float=False)
        }

    print("\n=== Final Best Params ===")
    print(best_params)
    return best_params

# ----------------------------
# Main pipeline
# ----------------------------
def run_pipeline(img_root, csv_file, out_model, N_slices_min, N_slices_max, *,
                 epochs=50, batch_size=8, device="auto", seed=42, save_test_preds=False):

    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    pairs = load_csv_pairs(csv_file, img_root)
    if len(pairs) == 0:
        raise RuntimeError("No image pairs loaded; aborting.")

    weights = np.array([w for _,w in pairs])
    n = len(pairs)
    idx = np.arange(n); np.random.shuffle(idx)
    val_n, test_n = max(1,int(0.2*n)), max(1,int(0.1*n))
    val_idx = idx[:val_n]
    test_idx = idx[val_n:val_n+test_n]
    train_idx = idx[val_n+test_n:]

    print(f"[info] Dataset sizes -> train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")

    # Precompute volumes for initial N_slices range
    vols_dict = {}
    for N_slices in range(N_slices_min, N_slices_max+1):
        vols = []
        for p in pairs:
            mask = image_to_mask(p[0])
            bbox = bbox_from_mask(mask)
            vols.append(estimate_volume_from_mask(mask, N_slices, bbox))
        vols_arr = np.array(vols).reshape(-1,1)
        vols_dict[N_slices] = vols_arr
        print(f"[info] Computed volumes for N_slices={N_slices} (shape {vols_arr.shape})")

    # Grid search with validation split (pass full vols_dict + indices)
    best_params = grid_search_cv(
        vols_dict,
        train_idx, val_idx, weights,
        device, batch_size
    )

    # Final volumes with chosen N_slices
    chosen_N = int(best_params.get("N_slices", N_slices_min))
    vols = vols_dict.get(chosen_N)
    if vols is None:
        vols = vols_dict[N_slices_min]
        chosen_N = N_slices_min

    # Build final train/test splits (use train_idx/test_idx determined above)
    Xtr = vols[train_idx].reshape(-1)
    ytr = weights[train_idx]
    Xte = vols[test_idx].reshape(-1)
    yte = weights[test_idx]

    # Train final model (full epochs)
    model, scaler, _ = train_one_model(Xtr, ytr, Xte, yte, int(best_params["degree"]),
                                       float(best_params["lr"]), float(best_params["alpha"]), float(best_params["l1_ratio"]),
                                       batch_size, epochs, device)

    torch.save({"state": model.state_dict(),
                "scaler_mean": scaler.mean_,
                "scaler_std": scaler.std_,
                "params": best_params}, out_model)
    print(f"Model saved to {out_model}")

    # ---- Compute test metrics ----
    metrics, preds = compute_metrics_and_preds(model, scaler, Xte, yte, int(best_params["degree"]), device)
    print("\n=== Test Set Metrics ===")
    print(f"MSE: {metrics['mse']:.4f}")
    print(f"MAE: {metrics['mae']:.4f}")
    print(f"R² : {metrics['r2']:.4f}")

    if save_test_preds:
        out_csv = os.path.splitext(out_model)[0] + "_test_preds.csv"
        with open(out_csv, "w", newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Image", "True_Weight", "Pred_Weight"])
            for i, idx_val in enumerate(test_idx):
                writer.writerow([pairs[idx_val][0], float(weights[idx_val]), float(preds[i])])
        print(f"Test predictions saved to {out_csv}")

# ----------------------------
# Hard-coded config
# ----------------------------
if __name__=="__main__":
    img_root = "/content/drive/MyDrive/Tomato Yield Estimation RE/Tomato-Images/Tomato"   # folder containing images referenced in CSV
    csv_file = "/content/drive/MyDrive/Tomato Yield Estimation RE/tomato.csv"  # CSV with columns: Image Name, Weight, View, Tomato ID
    out_model = "/content/drive/MyDrive/Tomato Yield Estimation RE/tomato_volume_model_gridsearch.pth"
    N_slices_min = 10
    N_slices_max = 15

    epochs = 20
    batch_size = 8
    device = "auto"
    seed = 42
    save_test_preds = True

    run_pipeline(img_root, csv_file, out_model, N_slices_min, N_slices_max,
                 epochs=epochs, batch_size=batch_size, device=device, seed=seed,
                 save_test_preds=save_test_preds)


[info] Loaded 45 valid image pairs from '/content/drive/MyDrive/Tomato Yield Estimation RE/tomato.csv'.
[info] Dataset sizes -> train: 32, val: 9, test: 4
[info] Computed volumes for N_slices=10 (shape (45, 1))
[info] Computed volumes for N_slices=11 (shape (45, 1))
[info] Computed volumes for N_slices=12 (shape (45, 1))
[info] Computed volumes for N_slices=13 (shape (45, 1))
[info] Computed volumes for N_slices=14 (shape (45, 1))
[info] Computed volumes for N_slices=15 (shape (45, 1))

--- Grid Search Round 1/3 ---
Best so far: {'lr': 0.1, 'alpha': 0.01, 'l1_ratio': 1.0, 'degree': 1, 'N_slices': 11} loss: 3370.2112630208335

--- Grid Search Round 2/3 ---
Best so far: {'lr': 0.2, 'alpha': 0.001, 'l1_ratio': 1.0, 'degree': 1, 'N_slices': 12} loss: 3166.419921875

--- Grid Search Round 3/3 ---
Best so far: {'lr': 0.4, 'alpha': 0.01, 'l1_ratio': 0.75, 'degree': 2, 'N_slices': 11} loss: 2723.3057454427085

=== Final Best Params ===
{'lr': 0.4, 'alpha': 0.01, 'l1_ratio': 0.75, 'degree': 2, 

In [ ]:
import cv2
import numpy as np
import os
from google.colab import drive

# Mount Google Drive
# drive.mount('/content/drive') # Assuming drive is already mounted from a previous cell

# Define paths
img_root = "/content/drive/MyDrive/Tomato Yield Estimation RE/Tomato-Images/Tomato" # Source folder
output_folder = "/content/drive/MyDrive/Tomato Yield Estimation RE/Tomato_Bbox_Color" # Output folder

# Create output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created output folder: {output_folder}")

# Get list of image files (assuming .jpg or .png)
image_files = [f for f in os.listdir(img_root) if f.endswith(('.JPG', '.jpeg', '.png'))]

print(f"Processing {len(image_files)} images...")

for img_name in image_files:
    img_path = os.path.join(img_root, img_name)
    output_path = os.path.join(output_folder, f"bbox_{img_name}")

    try:
        # Read the image
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not read image {img_name}. Skipping.")
            continue

        # Convert BGR to HSV
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

        # Define range for red color in HSV
        # Note: Red wraps around in HSV, so two ranges are needed.
        lower_red1 = np.array([0, 100, 100])
        upper_red1 = np.array([10, 255, 255])
        lower_red2 = np.array([160, 100, 100])
        upper_red2 = np.array([180, 255, 255])

        # Threshold the HSV image to get only red colors
        mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
        mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
        mask = mask1 + mask2

        # Find contours in the mask
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Draw bounding boxes around detected contours
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2) # Green bounding box

        # Save the image with bounding boxes
        cv2.imwrite(output_path, img)

    except Exception as e:
        print(f"An error occurred while processing {img_name}: {e}")

print("Processing complete.")

Processing 45 images...
Processing complete.


OpenCV for Bounding Box

In [ ]:
#!/usr/bin/env python3
"""
volume_to_weight_train_gridsearch.py

Modified version:
 - Uses OpenCV HSV color thresholding for mask and bounding box generation instead of Otsu.
 - Uses 3 rounds of Grid Search CV (instead of Bayesian optimization).
 - Each round runs 5 epochs to quickly evaluate hyperparameters.
 - Each round narrows the search space around the previous best.
 - Final training uses full epochs (default = 50).
 - After training, prints MSE, MAE, and R² on the test set.
"""

import os
import math
import csv
import random
from typing import List, Tuple, Dict, Any

import numpy as np
from PIL import Image
import cv2  # Added for HSV-based processing
from tqdm import tqdm # Added for progress visualization

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------------
# NEW: Mask / bbox using OpenCV HSV
# -----------------------------------
def get_hsv_mask_and_bbox(img_path: str) -> Tuple[np.ndarray, Tuple[int, int, int, int]]:
    """
    Generates a binary mask and a bounding box for the largest red object in an image
    using HSV color thresholding. Replaces the Otsu thresholding logic.
    """
    try:
        # Read the image with OpenCV
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not read image {img_path}. Skipping.")
            return np.zeros((1, 1), dtype=np.uint8), (0, 0, 0, 0)

        # Convert BGR to HSV
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

        # Define range for red color in HSV (two ranges for wrap-around)
        lower_red1 = np.array([0, 100, 100])
        upper_red1 = np.array([10, 255, 255])
        lower_red2 = np.array([160, 100, 100])
        upper_red2 = np.array([180, 255, 255])

        # Threshold the HSV image to get only red colors
        mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
        mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
        full_mask = mask1 + mask2

        # Find contours in the mask
        contours, _ = cv2.findContours(full_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            # Return empty mask and bbox if no contours are found
            return np.zeros_like(full_mask, dtype=np.uint8), (0, 0, 0, 0)

        # Find the largest contour by area
        largest_contour = max(contours, key=cv2.contourArea)

        # Get the bounding box of the largest contour
        x, y, w, h = cv2.boundingRect(largest_contour)

        # Convert bbox to (minx, miny, maxx, maxy) format for compatibility
        bbox = (x, y, x + w, y + h)

        # The final mask must be binary (0 or 1) for volume estimation
        binary_mask = (full_mask > 0).astype(np.uint8)

        return binary_mask, bbox

    except Exception as e:
        print(f"Warning: Error processing {img_path}: {e}. Returning empty mask/bbox.")
        return np.zeros((1, 1), dtype=np.uint8), (0, 0, 0, 0)

# ----------------------------
# Mask / bbox / volume utils (Unchanged)
# ----------------------------
def avg_row_width(mask: np.ndarray, y: int, minx: int, maxx: int, half_window: int = 1) -> float:
    H, W = mask.shape
    ys = range(max(0, y-half_window), min(H, y+half_window+1))
    widths = [int(np.sum(mask[yy, minx:maxx+1] > 0)) for yy in ys]
    return float(np.mean(widths)) if widths else 0.0

def estimate_volume_from_mask(mask: np.ndarray, N_slices: int, bbox: Tuple[int,int,int,int], boundary_window: int = 1) -> float:
    minx, miny, maxx, maxy = bbox
    if minx == maxx and miny == maxy:
        return 0.0

    L = maxy - miny + 1
    if L <= 0:
        return 0.0
    slice_h = float(L) / float(N_slices)

    def area_at_row(yf: float) -> float:
        y = int(round(yf))
        y = max(miny, min(maxy, y))
        w = avg_row_width(mask, y, minx, maxx, boundary_window)
        return (math.pi / 4.0) * (w ** 2)

    total_vol = 0.0
    for i in range(N_slices):
        top_f = miny + i * slice_h
        bottom_f = miny + (i + 1) * slice_h
        h = bottom_f - top_f
        A_top = area_at_row(top_f)
        A_bottom = area_at_row(bottom_f)
        if i == 0:
            V = (h / 3.0) * A_bottom
        elif i == N_slices - 1:
            V = (h / 3.0) * A_top
        else:
            V = (h / 3.0) * (A_top + A_bottom + math.sqrt(max(0.0, A_top * A_bottom)))
        total_vol += V
    return total_vol

# ----------------------------
# CSV loader (Unchanged)
# ----------------------------
def load_csv_pairs(csv_file: str, img_root: str = "") -> List[Tuple[str, float]]:
    pairs = []
    missing = []
    img_root_norm = os.path.normpath(img_root) if img_root else ""

    with open(csv_file, "r", newline='') as f:
        reader = csv.reader(f)
        header = next(reader, None)
        if header and len(header) >= 2:
            has_header = "image" in header[0].lower() or "weight" in header[1].lower()
        else:
            has_header = False

        def resolve_path(image_name: str) -> str:
            image_name = image_name.strip().strip('"').strip("'")
            pth = os.path.join(img_root_norm, image_name)
            if os.path.exists(pth):
                return os.path.normpath(pth)
            return ""

        if not has_header and header:
            # header actually contains the first data row
            try:
                img_p = resolve_path(header[0])
                wt = float(header[1])
                if img_p:
                    pairs.append((img_p, wt))
                else:
                    missing.append(header[0])
            except Exception:
                missing.append(header[0])

        for row in reader:
            if len(row) < 2:
                continue
            img_name = row[0]
            try:
                wt = float(row[1])
            except:
                try:
                    wt = float(row[1].strip())
                except:
                    missing.append(img_name)
                    continue
            img_p = resolve_path(img_name)
            if img_p:
                pairs.append((img_p, wt))
            else:
                missing.append(img_name)

    if missing:
        print(f"[warning] {len(missing)} image paths in CSV could not be resolved - skipped.")
    print(f"[info] Loaded {len(pairs)} valid image pairs from '{csv_file}'.")
    return pairs

# ----------------------------
# Features & model (Unchanged)
# ----------------------------
class SimpleScaler:
    def fit(self, X):
        self.mean_, self.std_ = np.mean(X,0,keepdims=True), np.std(X,0,keepdims=True)
        self.std_[self.std_==0]=1
    def transform(self, X): return (X - self.mean_) / self.std_
    def fit_transform(self, X): self.fit(X); return self.transform(X)

def poly_features(vols: np.ndarray, degree: int) -> np.ndarray:
    vols = np.array(vols).reshape(-1,1)
    return np.column_stack([vols[:,0]**d for d in range(1, degree+1)])

class PolyRegModel(nn.Module):
    def __init__(self, in_features):
        super().__init__(); self.lin = nn.Linear(in_features, 1)
    def forward(self, x): return self.lin(x).squeeze(1)

# ----------------------------
# Training routine (Unchanged)
# ----------------------------
def train_one_model(X_train, y_train, X_val, y_val, degree, lr, alpha, l1_ratio, batch_size, epochs, device):
    X_train = np.array(X_train).reshape(-1)
    X_val = np.array(X_val).reshape(-1)
    y_train = np.array(y_train).reshape(-1)
    y_val = np.array(y_val).reshape(-1)

    X_train_f = poly_features(X_train, degree)
    X_val_f = poly_features(X_val, degree)
    scaler = SimpleScaler()
    X_train_s = scaler.fit_transform(X_train_f)
    X_val_s = scaler.transform(X_val_f)

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train_s,dtype=torch.float32), torch.tensor(y_train,dtype=torch.float32)),
        batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.tensor(X_val_s,dtype=torch.float32), torch.tensor(y_val,dtype=torch.float32)),
        batch_size=max(1,batch_size//2), shuffle=False
    )

    model = PolyRegModel(degree).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss, best_state = float('inf'), None
    for _ in range(epochs):
        model.train()
        for xb,yb in train_loader:
            xb,yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            preds = model(xb)
            mse = ((preds - yb)**2).mean()
            l1 = sum(p.abs().sum() for p in model.parameters())
            l2 = sum((p**2).sum() for p in model.parameters())
            loss = mse + alpha*(l1_ratio*l1 + (1-l1_ratio)*0.5*l2)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = 0.0
            for xb,yb in val_loader:
                xb,yb = xb.to(device), yb.to(device)
                preds = model(xb)
                mse = ((preds - yb)**2).mean()
                l1 = sum(p.abs().sum() for p in model.parameters())
                l2 = sum((p**2).sum() for p in model.parameters())
                val_loss += (mse + alpha*(l1_ratio*l1 + (1-l1_ratio)*0.5*l2)).item()
            val_loss /= max(1, len(val_loader))
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    return model, scaler, best_loss

# ----------------------------
# Metrics computation (Unchanged)
# ----------------------------
def compute_metrics_and_preds(model, scaler, X, y, degree, device):
    X_f = poly_features(np.array(X).reshape(-1,1), degree)
    X_s = scaler.transform(X_f)
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_s,dtype=torch.float32).to(device)).cpu().numpy()
    mse = ((preds - y)**2).mean()
    mae = np.mean(np.abs(preds - y))
    ss_res = ((y - preds) ** 2).sum()
    ss_tot = ((y - np.mean(y)) ** 2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return {"mse": mse, "mae": mae, "r2": r2}, preds

# ----------------------------
# Grid search (Unchanged)
# ----------------------------
def grid_search_cv(vols_dict: Dict[int, np.ndarray],
                   train_idx: np.ndarray, val_idx: np.ndarray, weights: np.ndarray,
                   device, batch_size, rounds=3, epochs_per_round=5) -> Dict[str, Any]:
    param_grid = {
        "lr": [1e-3, 1e-2, 1e-1],
        "alpha": [1e-6, 1e-4, 1e-2],
        "l1_ratio": [0.0, 0.5, 1.0],
        "degree": [1, 2, 3, 4, 5],
        "N_slices": sorted(list(vols_dict.keys()))
    }

    best_params = None
    for r in range(rounds):
        print(f"\n--- Grid Search Round {r+1}/{rounds} ---")
        best_loss = float("inf")
        # iterate grid
        for lr in param_grid["lr"]:
            for alpha in param_grid["alpha"]:
                for l1_ratio in param_grid["l1_ratio"]:
                    for degree in param_grid["degree"]:
                        for N_slices in param_grid["N_slices"]:
                            vols_all = vols_dict.get(N_slices)
                            if vols_all is None:
                                continue
                            X_train = vols_all[train_idx].reshape(-1)
                            y_train = weights[train_idx]
                            X_val = vols_all[val_idx].reshape(-1)
                            y_val = weights[val_idx]

                            try:
                                model, scaler, vloss = train_one_model(
                                    X_train, y_train, X_val, y_val,
                                    degree, lr, alpha, l1_ratio,
                                    batch_size, epochs_per_round, device
                                )
                            except AssertionError as e:
                                print(f"[warn] skipped config lr={lr},alpha={alpha},l1_ratio={l1_ratio},degree={degree},N_slices={N_slices} due to error: {e}")
                                continue

                            if vloss < best_loss:
                                best_loss = vloss
                                best_params = {
                                    "lr": lr,
                                    "alpha": alpha,
                                    "l1_ratio": l1_ratio,
                                    "degree": degree,
                                    "N_slices": N_slices
                                }
        print("Best so far:", best_params, "loss:", best_loss)

        if best_params is None:
            print("[error] No valid params found in grid search; returning defaults.")
            return {
                "lr": 1e-3, "alpha": 1e-6, "l1_ratio": 0.0, "degree": 2,
                "N_slices": sorted(list(vols_dict.keys()))[0]
            }

        # Narrow grid for next round
        def narrow(values, best, is_float=False, factor=2.0):
            if is_float:
                candidates = sorted(set([max(1e-12, best/factor), best, best*factor]))
                return candidates
            else:
                best_i = int(best)
                return sorted(set([max(1,best_i-1), best_i, best_i+1]))

        param_grid = {
            "lr": narrow(param_grid["lr"], best_params["lr"], is_float=True, factor=2.0),
            "alpha": narrow(param_grid["alpha"], best_params["alpha"], is_float=True, factor=10.0),
            "l1_ratio": sorted(set([
                max(0.0, best_params["l1_ratio"]-0.25),
                best_params["l1_ratio"],
                min(1.0, best_params["l1_ratio"]+0.25)
            ])),
            "degree": narrow(param_grid["degree"], best_params["degree"], is_float=False),
            "N_slices": narrow(param_grid["N_slices"], best_params["N_slices"], is_float=False)
        }

    print("\n=== Final Best Params ===")
    print(best_params)
    return best_params

# ----------------------------
# Main pipeline (MODIFIED)
# ----------------------------
def run_pipeline(img_root, csv_file, out_model, N_slices_min, N_slices_max, *,
                 epochs=50, batch_size=8, device="auto", seed=42, save_test_preds=False):

    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    pairs = load_csv_pairs(csv_file, img_root)
    if len(pairs) == 0:
        raise RuntimeError("No image pairs loaded; aborting.")

    weights = np.array([w for _,w in pairs])
    n = len(pairs)
    idx = np.arange(n); np.random.shuffle(idx)
    val_n, test_n = max(1,int(0.2*n)), max(1,int(0.1*n))
    val_idx = idx[:val_n]
    test_idx = idx[val_n:val_n+test_n]
    train_idx = idx[val_n+test_n:]

    print(f"[info] Dataset sizes -> train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")

    # Pre-compute masks and bounding boxes once for all images using the new method
    print("[info] Pre-computing masks and bounding boxes for all images...")
    precomputed_data = []
    for p in tqdm(pairs, desc="Processing Images"):
        mask, bbox = get_hsv_mask_and_bbox(p[0])
        precomputed_data.append({'mask': mask, 'bbox': bbox})

    # Precompute volumes for initial N_slices range using the precomputed data
    vols_dict = {}
    for N_slices in range(N_slices_min, N_slices_max + 1):
        vols = []
        for data in precomputed_data:
            mask = data['mask']
            bbox = data['bbox']
            vols.append(estimate_volume_from_mask(mask, N_slices, bbox))
        vols_arr = np.array(vols).reshape(-1, 1)
        vols_dict[N_slices] = vols_arr
        print(f"[info] Computed volumes for N_slices={N_slices} (shape {vols_arr.shape})")


    # Grid search with validation split (pass full vols_dict + indices)
    best_params = grid_search_cv(
        vols_dict,
        train_idx, val_idx, weights,
        device, batch_size
    )

    # Final volumes with chosen N_slices
    chosen_N = int(best_params.get("N_slices", N_slices_min))
    vols = vols_dict.get(chosen_N)
    if vols is None:
        vols = vols_dict[N_slices_min]
        chosen_N = N_slices_min

    # Build final train/test splits (use train_idx/test_idx determined above)
    Xtr = vols[train_idx].reshape(-1)
    ytr = weights[train_idx]
    Xte = vols[test_idx].reshape(-1)
    yte = weights[test_idx]

    # Train final model (full epochs)
    model, scaler, _ = train_one_model(Xtr, ytr, Xte, yte, int(best_params["degree"]),
                                       float(best_params["lr"]), float(best_params["alpha"]), float(best_params["l1_ratio"]),
                                       batch_size, epochs, device)

    torch.save({"state": model.state_dict(),
                "scaler_mean": scaler.mean_,
                "scaler_std": scaler.std_,
                "params": best_params}, out_model)
    print(f"Model saved to {out_model}")

    # ---- Compute test metrics ----
    metrics, preds = compute_metrics_and_preds(model, scaler, Xte, yte, int(best_params["degree"]), device)
    print("\n=== Test Set Metrics ===")
    print(f"MSE: {metrics['mse']:.4f}")
    print(f"MAE: {metrics['mae']:.4f}")
    print(f"R² : {metrics['r2']:.4f}")

    if save_test_preds:
        out_csv = os.path.splitext(out_model)[0] + "_test_preds.csv"
        with open(out_csv, "w", newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Image", "True_Weight", "Pred_Weight"])
            for i, idx_val in enumerate(test_idx):
                writer.writerow([pairs[idx_val][0], float(weights[idx_val]), float(preds[i])])
        print(f"Test predictions saved to {out_csv}")

# ----------------------------
# Hard-coded config
# ----------------------------
if __name__=="__main__":
    img_root = "/content/drive/MyDrive/Tomato Yield Estimation RE/Tomato-Images/Tomato"  # folder containing images referenced in CSV
    csv_file = "/content/drive/MyDrive/Tomato Yield Estimation RE/tomato.csv"  # CSV with columns: Image Name, Weight, View, Tomato ID
    out_model = "/content/drive/MyDrive/Tomato Yield Estimation RE/tomato_volume_model_gridsearch.pth"
    N_slices_min = 10
    N_slices_max = 15

    epochs = 20
    batch_size = 8
    device = "auto"
    seed = 42
    save_test_preds = True

    run_pipeline(img_root, csv_file, out_model, N_slices_min, N_slices_max,
                 epochs=epochs, batch_size=batch_size, device=device, seed=seed,
                 save_test_preds=save_test_preds)

[info] Loaded 45 valid image pairs from '/content/drive/MyDrive/Tomato Yield Estimation RE/tomato.csv'.
[info] Dataset sizes -> train: 32, val: 9, test: 4
[info] Pre-computing masks and bounding boxes for all images...


Processing Images: 100%|██████████| 45/45 [00:07<00:00,  5.98it/s]


[info] Computed volumes for N_slices=10 (shape (45, 1))
[info] Computed volumes for N_slices=11 (shape (45, 1))
[info] Computed volumes for N_slices=12 (shape (45, 1))
[info] Computed volumes for N_slices=13 (shape (45, 1))
[info] Computed volumes for N_slices=14 (shape (45, 1))
[info] Computed volumes for N_slices=15 (shape (45, 1))

--- Grid Search Round 1/3 ---
Best so far: {'lr': 0.1, 'alpha': 0.01, 'l1_ratio': 0.0, 'degree': 1, 'N_slices': 12} loss: 3426.6806640625

--- Grid Search Round 2/3 ---
Best so far: {'lr': 0.2, 'alpha': 0.001, 'l1_ratio': 0.0, 'degree': 1, 'N_slices': 11} loss: 3255.7799479166665

--- Grid Search Round 3/3 ---
Best so far: {'lr': 0.4, 'alpha': 0.001, 'l1_ratio': 0.25, 'degree': 1, 'N_slices': 10} loss: 2970.01171875

=== Final Best Params ===
{'lr': 0.4, 'alpha': 0.001, 'l1_ratio': 0.25, 'degree': 1, 'N_slices': 10}
Model saved to /content/drive/MyDrive/Tomato Yield Estimation RE/tomato_volume_model_gridsearch.pth

=== Test Set Metrics ===
MSE: 1451.5863
